In [2]:
# =================================================================
# NOTEBOOK: 02_feature_engineering.ipynb
# PHASE 3: DATA PREPARATION & FEATURE ENGINEERING
# GOAL: Create time-series features (lags, rolling stats), handle initial data cleanup,
# and perform initial dimensionality reduction.
# =================================================================

import pandas as pd
import numpy as np
import os

# --- 1. File Paths and Setup ---
# Project base path (using the user's path)
BASE_PATH = 'C:\\Users\\acer\\OneDrive\\Desktop\\AimlWebforecasting'
DATA_PROCESSED_INPUT_PATH = f'{BASE_PATH}\\data\\processed\\cleaned_data.pkl'
DATA_PROCESSED_OUTPUT_PATH = f'{BASE_PATH}\\data\\processed\\features_df.pkl'

# Load the cleaned dataset (output from 01_data_exploration.ipynb)
try:
    df = pd.read_pickle(DATA_PROCESSED_INPUT_PATH)
    print("--- Cleaned data loaded successfully ---")
except FileNotFoundError:
    print(f"Error: File not found at {DATA_PROCESSED_INPUT_PATH}. Run 01_data_exploration.ipynb first.")
    exit()

# --- 2. Data Cleaning and Imputation ---

# Check for and handle any remaining missing values created by time-series processing (if any)
# Use forward fill for time-series consistency, followed by median for any starting NaNs
if df.isnull().values.any():
    print("Handling missing values...")
    df.fillna(method='ffill', inplace=True)
    df.fillna(df.median(numeric_only=True), inplace=True)
else:
    print("No missing values found prior to feature engineering.")

# --- 3. Feature Engineering: Time-Based Features (Dimensionality Technique 1) ---

# Extract simple temporal features from the index
df['year'] = df.index.year
df['month'] = df.index.month
df['day'] = df.index.day
df['dow'] = df.index.dayofweek # Day of Week (0=Monday, 6=Sunday)
df['week_of_year'] = df.index.isocalendar().week.astype(int)

# Create a simple weekend indicator
df['is_weekend'] = df['dow'].apply(lambda x: 1 if x >= 5 else 0)


# --- 4. Feature Engineering: Lagged and Rolling Features (Time Series Core) ---

# Define the lag periods to use
LAG_PERIODS = [1, 2, 7, 30] # Traffic from yesterday, 2 days ago, last week, and last month
TARGET_COLUMN = 'y'

for lag in LAG_PERIODS:
    # Create lag features for the target variable (y)
    df[f'{TARGET_COLUMN}_lag_{lag}'] = df[TARGET_COLUMN].shift(lag)

# Create rolling window features (Rolling Mean is often a powerful predictor)
ROLLING_WINDOW = 7 # 7-day rolling mean
df[f'{TARGET_COLUMN}_rolling_mean_{ROLLING_WINDOW}'] = (
    df[TARGET_COLUMN].shift(1).rolling(window=ROLLING_WINDOW).mean()
)


# --- 5. Initial Dimensionality Reduction & Leakage Handling (Technique 2) ---

# Drop features highly correlated with the target (y) that represent potential data leakage
# or are redundant (Multicollinearity/Leakage handling)

DROP_COLUMNS = [
    'page_views',          # Highly correlated and effectively a proxy for 'y'
    'new_users',           # Likely correlated with 'y' and conversions
    'returning_users',     # Likely correlated with 'y' and conversions
    'conversions',         # Extreme leakage (conversion count)
    'conversion_rate'      # Extreme leakage (derived from conversions)
]

# We also drop the original promo_flag and holiday_flag as they are already in the df
# and will be used as categorical features later, but we ensure the highly coupled
# variables are dropped here.

df.drop(columns=DROP_COLUMNS, errors='ignore', inplace=True)
print(f"\nDropped leakage/redundant columns: {DROP_COLUMNS}")


# --- 6. Final Clean Up after Lags/Rolling Stats ---

# Lag and rolling features introduce NaN values at the start of the series.
# Drop rows with NaN values. This results in a slightly shorter, but clean, dataset.
df.dropna(inplace=True)
print(f"Dataset shape after dropping NaNs from lags/rolling windows: {df.shape}")
print(f"Number of rows lost ({max(LAG_PERIODS)}): {len(df) - df.shape[0]}")

# --- 7. Output Final Feature Matrix ---

# Save the final DataFrame containing all engineered features
df.to_pickle(DATA_PROCESSED_OUTPUT_PATH)
print(f"\nFinal feature matrix saved to '{DATA_PROCESSED_OUTPUT_PATH}' for 03_modeling.ipynb.")

# --- Inspection ---
print("\n--- Final Features Snapshot ---")
print(df.head())

# =================================================================

--- Cleaned data loaded successfully ---
No missing values found prior to feature engineering.

Dropped leakage/redundant columns: ['page_views', 'new_users', 'returning_users', 'conversions', 'conversion_rate']
Dataset shape after dropping NaNs from lags/rolling windows: (1470, 22)
Number of rows lost (30): 0

Final feature matrix saved to 'C:\Users\acer\OneDrive\Desktop\AimlWebforecasting\data\processed\features_df.pkl' for 03_modeling.ipynb.

--- Final Features Snapshot ---
               y  clicks  impressions     ctr  bounce_rate  \
date                                                         
2020-01-31  2272     159         3411  0.0466       0.1870   
2020-02-01  2366     100         2663  0.0376       0.2947   
2020-02-02  2244     133         3391  0.0392       0.2715   
2020-02-03  2373     123         3422  0.0359       0.4089   
2020-02-04  2931     194         3990  0.0486       0.3595   

            avg_session_duration_sec  ad_spend_usd  organic_search  \
date         